# Credential Dumping: Browser Credential Storage

| Field | Value |
|---|---|
| MITRE ATT&CK | T1003 |
| Tactic | Credential Access |
| Difficulty | Intermediate |
| Time | ~30 min |

## Threat Context

Raccoon Stealer, first observed in 2019 and resurgent as Raccoon Stealer v2 in 2022, became one of the most widely deployed information-stealing malware families. Sold as a Malware-as-a-Service (MaaS) offering for as little as $200/month, it specialized in extracting stored credentials from web browsers, cryptocurrency wallets, and email clients.

The stealer targeted browser credential databases directly. Modern browsers like Chrome, Firefox, and Edge store saved passwords in SQLite databases. While the credentials are encrypted, the encryption keys are often stored locally and accessible to any process running under the same user context. Raccoon Stealer automated the extraction of these databases, decrypted the credentials, and exfiltrated them to C2 servers.

The U.S. Department of Justice indicted the developer of Raccoon Stealer in 2022, noting that the malware had compromised over 50 million credentials worldwide. The case highlighted how simple it is to access browser credential storage — the same databases that conveniently auto-fill your passwords are trivially readable by any malware running with user-level privileges.

## What You'll Learn
- How browsers store credentials in SQLite databases
- How info-stealers extract and parse credential data
- How to detect credential access through database monitoring

In [ ]:
# EDUCATIONAL PURPOSE ONLY
# Demonstrates browser credential storage concepts using a DUMMY database
# Does NOT access any real browser credential stores

import sqlite3
import os
import json
import hashlib
import base64
import tempfile
from datetime import datetime
from pathlib import Path

SANDBOX = tempfile.mkdtemp(prefix='cyberlab_creds_')
print(f'Sandbox: {SANDBOX}')
print('Using DUMMY data only — no real credentials are accessed.')

## 1. Browser Credential Storage Architecture

Browsers store credentials in SQLite databases with well-known schemas. Chrome uses `Login Data`, Firefox uses `logins.json` backed by `key4.db`. The database locations are predictable, which makes them easy targets for stealers.

In [ ]:
# EDUCATIONAL PURPOSE ONLY
# Create a DUMMY browser credential database matching Chrome's schema

DUMMY_DB = os.path.join(SANDBOX, 'Login Data')

def create_dummy_credential_db(db_path):
    """Create a SQLite database mimicking Chrome's credential storage schema."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # Chrome's actual schema (simplified)
    cursor.execute('''
        CREATE TABLE logins (
            origin_url TEXT NOT NULL,
            action_url TEXT,
            username_element TEXT,
            username_value TEXT,
            password_element TEXT,
            password_value BLOB,
            date_created INTEGER NOT NULL,
            date_last_used INTEGER,
            times_used INTEGER DEFAULT 0
        )
    ''')
    
    # Insert DUMMY credentials (fake domains, fake passwords)
    dummy_creds = [
        ('https://mail.example.com/login', 'https://mail.example.com/auth',
         'email', 'alice@example.com', 'password',
         base64.b64encode(b'ENCRYPTED_dummy_pass_1'), 13300000000, 13310000000, 45),
        ('https://bank.example.com/', 'https://bank.example.com/signin',
         'user', 'alice_banking', 'pass',
         base64.b64encode(b'ENCRYPTED_dummy_pass_2'), 13290000000, 13310000000, 12),
        ('https://vpn.example.com/', 'https://vpn.example.com/login',
         'username', 'acorp\\alice', 'password',
         base64.b64encode(b'ENCRYPTED_dummy_pass_3'), 13280000000, 13305000000, 89),
        ('https://social.example.com/', 'https://social.example.com/login',
         'login', 'alice_social', 'passwd',
         base64.b64encode(b'ENCRYPTED_dummy_pass_4'), 13250000000, 13300000000, 200),
        ('https://dev.example.com/', 'https://dev.example.com/session',
         'login_field', 'alice-dev', 'password_field',
         base64.b64encode(b'ENCRYPTED_dummy_pass_5'), 13270000000, 13308000000, 30),
    ]
    
    cursor.executemany(
        'INSERT INTO logins VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)',
        dummy_creds
    )
    
    conn.commit()
    conn.close()
    return db_path

create_dummy_credential_db(DUMMY_DB)
print(f'Dummy credential database created: {DUMMY_DB}')
print(f'Database size: {os.path.getsize(DUMMY_DB)} bytes')

# Show known credential store locations (for awareness)
print('\n--- Real Browser Credential Locations (DO NOT ACCESS) ---')
locations = {
    'Chrome (Win)': r'%LOCALAPPDATA%\Google\Chrome\User Data\Default\Login Data',
    'Chrome (macOS)': '~/Library/Application Support/Google/Chrome/Default/Login Data',
    'Firefox (Win)': r'%APPDATA%\Mozilla\Firefox\Profiles\*.default\logins.json',
    'Edge (Win)': r'%LOCALAPPDATA%\Microsoft\Edge\User Data\Default\Login Data',
}
for browser, path in locations.items():
    print(f'  {browser}: {path}')

## 2. How Info-Stealers Extract Credentials

Info-stealers follow a predictable pattern: copy the database (to avoid lock conflicts with the browser), query the logins table, and attempt to decrypt the password blobs. We demonstrate the extraction logic against our dummy database.

In [ ]:
# EDUCATIONAL PURPOSE ONLY
# Demonstrate credential extraction from DUMMY database

import shutil

def extract_credentials(db_path):
    """Extract credentials from a Chrome-style Login Data database.
    Works on our DUMMY database only."""
    
    # Step 1: Copy database (stealers do this to avoid browser lock)
    temp_copy = db_path + '.tmp'
    shutil.copy2(db_path, temp_copy)
    
    # Step 2: Query the logins table
    conn = sqlite3.connect(temp_copy)
    cursor = conn.cursor()
    cursor.execute('''
        SELECT origin_url, username_value, password_value,
               date_created, date_last_used, times_used
        FROM logins
        ORDER BY times_used DESC
    ''')
    
    results = []
    for row in cursor.fetchall():
        url, username, encrypted_pass, created, last_used, times = row
        
        # Step 3: In real attacks, decryption happens here using:
        # - Windows DPAPI (CryptUnprotectData)
        # - Chrome's Local State key (AES-GCM with DPAPI-protected key)
        # We just decode our dummy base64 data
        try:
            decoded = base64.b64decode(encrypted_pass).decode('utf-8')
            password_display = '[ENCRYPTED: ' + decoded[:20] + '...]'
        except Exception:
            password_display = '[ENCRYPTED BLOB]'
        
        results.append({
            'url': url,
            'username': username,
            'password': password_display,
            'times_used': times,
            'value_score': _score_credential(url, times),
        })
    
    conn.close()
    os.remove(temp_copy)
    return results

def _score_credential(url, times_used):
    """Score a credential's value to an attacker."""
    high_value_keywords = ['bank', 'vpn', 'admin', 'mail', 'corp']
    score = min(times_used / 10, 5)  # Usage frequency
    for kw in high_value_keywords:
        if kw in url.lower():
            score += 3
    return round(score, 1)

extracted = extract_credentials(DUMMY_DB)

print(f'=== Extracted Credentials (DUMMY DATA) ===')
print(f'Total credentials found: {len(extracted)}\n')
print(f'{"URL":<35s} {"Username":<20s} {"Uses":>5s} {"Value":>6s}')
print('-' * 70)
for cred in extracted:
    url_short = cred['url'][:33] + '..' if len(cred['url']) > 35 else cred['url']
    print(f"{url_short:<35s} {cred['username']:<20s} {cred['times_used']:>5d} {cred['value_score']:>6.1f}")

## 3. Credential Access Detection

Monitoring access to browser credential databases is a key detection strategy. Any process other than the browser itself accessing these files should trigger an alert.

In [ ]:
# EDUCATIONAL PURPOSE ONLY
# Simulate credential access monitoring

class CredentialAccessMonitor:
    """Monitor and detect unauthorized credential database access."""
    
    def __init__(self):
        self.allowed_processes = {
            'chrome.exe', 'firefox.exe', 'msedge.exe',
            'Google Chrome', 'Firefox', 'Microsoft Edge',
        }
        self.alerts = []
        self.access_log = []
    
    def log_access(self, process_name, file_path, access_type):
        """Log a file access event and check if it's suspicious."""
        event = {
            'timestamp': datetime.now().isoformat(),
            'process': process_name,
            'file': file_path,
            'access_type': access_type,
        }
        self.access_log.append(event)
        
        # Check if access is authorized
        is_cred_file = any(kw in file_path.lower()
                          for kw in ['login data', 'logins.json', 'key4.db', 'cookies'])
        is_authorized = process_name in self.allowed_processes
        
        if is_cred_file and not is_authorized:
            alert = {
                'timestamp': event['timestamp'],
                'severity': 'CRITICAL',
                'type': 'UNAUTHORIZED_CREDENTIAL_ACCESS',
                'process': process_name,
                'file': file_path,
                'access_type': access_type,
                'mitre': 'T1003',
            }
            self.alerts.append(alert)
            return alert
        return None

# Simulate access events
monitor = CredentialAccessMonitor()

# Normal browser access (no alert)
monitor.log_access('chrome.exe', 'C:\\Users\\alice\\AppData\\Local\\Google\\Chrome\\User Data\\Default\\Login Data', 'READ')

# Suspicious access patterns (mimicking Raccoon Stealer behavior)
suspicious_accesses = [
    ('svchost_helper.exe', 'C:\\Users\\alice\\AppData\\Local\\Google\\Chrome\\User Data\\Default\\Login Data', 'COPY'),
    ('update_service.exe', 'C:\\Users\\alice\\AppData\\Roaming\\Mozilla\\Firefox\\Profiles\\abc.default\\logins.json', 'READ'),
    ('svchost_helper.exe', 'C:\\Users\\alice\\AppData\\Local\\Microsoft\\Edge\\User Data\\Default\\Login Data', 'COPY'),
    ('rundll32.exe', 'C:\\Users\\alice\\AppData\\Local\\Google\\Chrome\\User Data\\Default\\Cookies', 'READ'),
]

print('=== Credential Access Monitor ===')
for proc, path, access in suspicious_accesses:
    alert = monitor.log_access(proc, path, access)
    if alert:
        file_short = os.path.basename(alert['file'])
        print(f"[{alert['severity']}] {alert['process']} -> {file_short} ({alert['access_type']})")

print(f'\nTotal access events: {len(monitor.access_log)}')
print(f'Alerts generated: {len(monitor.alerts)}')

In [ ]:
# EDUCATIONAL PURPOSE ONLY
# Visualization: Credential value analysis and access patterns

import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0F0F0F')
fig.suptitle('Credential Dumping Analysis', color='#EDEAE4', fontsize=14, y=1.02)

# Left: Credential value scores
ax1.set_facecolor('#1A1A1A')
urls = [c['url'].split('//')[1].split('/')[0] for c in extracted]
scores = [c['value_score'] for c in extracted]
colors = ['#CF6C6C' if s >= 5 else '#CF9B6C' if s >= 3 else '#6C9BCF' for s in scores]
bars = ax1.barh(urls, scores, color=colors, height=0.5, edgecolor='#282828')
ax1.set_xlabel('Attacker Value Score', color='#7A7570')
ax1.set_title('Credential Value to Attackers', color='#EDEAE4', fontsize=12)
ax1.tick_params(colors='#7A7570')
for spine in ax1.spines.values():
    spine.set_edgecolor('#282828')

# Right: Access events - authorized vs unauthorized
ax2.set_facecolor('#1A1A1A')
auth_count = len(monitor.access_log) - len(monitor.alerts)
unauth_count = len(monitor.alerts)
ax2.pie([auth_count, unauth_count],
        labels=['Authorized', 'Unauthorized'],
        colors=['#6C9BCF', '#CF6C6C'],
        autopct='%1.0f%%',
        textprops={'color': '#EDEAE4', 'fontsize': 11},
        wedgeprops={'edgecolor': '#282828'})
ax2.set_title('Credential DB Access Events', color='#EDEAE4', fontsize=12)

plt.tight_layout()
plt.show()

## Defender's Perspective

| Indicator | Type | Detection |
|---|---|---|
| Non-browser process accessing Login Data / logins.json | File Access | Sysmon Event ID 11 (FileCreate) with path filters |
| SQLite queries against credential tables from unexpected processes | API Call | EDR behavioral detection on SQLite library loading |
| Mass credential file copying across browser profiles | File Operation | SIEM correlation on file copy events to temp directories |
| DPAPI CryptUnprotectData calls from non-browser processes | API Call | Windows Event ID 4693 (DPAPI activity) |

## Article Seed

**Title:** How Raccoon Stealer Harvests 50 Million Passwords from Your Browser

**Opening:** Your browser's password manager is a convenience feature that doubles as an attacker's treasure chest. Raccoon Stealer demonstrated that extracting saved credentials requires nothing more than copying a SQLite database and calling a Windows API — no privilege escalation needed.

**Sections:**
1. Inside Chrome's credential storage: SQLite and DPAPI
2. The Raccoon Stealer extraction pipeline
3. Building a credential access monitor in Python
4. Why password managers with master keys beat browser storage

**Tags:** cybersecurity, credential-theft, browser-security, sqlite, python

In [ ]:
# EDUCATIONAL PURPOSE ONLY
# Self-check assertions

import shutil

# Verify dummy database was created with correct schema
conn = sqlite3.connect(DUMMY_DB)
cursor = conn.cursor()
cursor.execute('SELECT COUNT(*) FROM logins')
count = cursor.fetchone()[0]
assert count == 5, f'Expected 5 dummy credentials, got {count}'
conn.close()

# Verify extraction worked
assert len(extracted) == 5, f'Expected 5 extracted credentials, got {len(extracted)}'

# Verify scoring assigns high value to banking/vpn
bank_cred = next(c for c in extracted if 'bank' in c['url'])
assert bank_cred['value_score'] >= 3, 'Banking credentials should be high value'

# Verify monitoring detected unauthorized access
assert len(monitor.alerts) == 4, f'Expected 4 alerts, got {len(monitor.alerts)}'
for alert in monitor.alerts:
    assert alert['severity'] == 'CRITICAL'
    assert alert['process'] not in monitor.allowed_processes

# Cleanup
shutil.rmtree(SANDBOX, ignore_errors=True)
print('All assertions passed.')
print('Sandbox cleaned up.')